In [ ]:
import glob
import sys
from itertools import batched
from pathlib import Path
from warnings import warn

from PIL import Image
from IPython.display import display

In [ ]:
# Use this module without installing it.
sys.path.insert(0, "./src")
from jetpyck.gfxdat import *

# These are not exported by default:
from jetpyck.gfxdat import jswitch_name_decode, jswitch_name_encode

In [ ]:
BASEDIR = Path("./tilesets/")

In [ ]:
for f in sorted(BASEDIR.glob("_jetp_?.dat", case_sensitive=False), key=lambda f: f.name.lower()):
    data = f.read_bytes()
    print("{:>12} {!r:36} = {!r}".format(f.name, jswitch_name_decode(data), data))

---

I could probably post this research to https://tcrf.net/ and/or https://datacrystal.tcrf.net/ and/or https://moddingwiki.shikadi.net/wiki/

But first I'll write everything down and save on GitHub.

---

In [ ]:
# XXX: This is a hacky experiment! NOT ready yet!
# The xjetpack uses a different format for the graphics.
# I don't know this format yet.
# This code here is just a quick copy-and-paste, for quick experimentation.
def xmas_gfxdat_parser(rawdata: bytes):
    # Raw palette has one byte per channel (R, G, B).
    # Each channel is 6-bit (0..63), as per VGA palette limitation.
    #rawpalette = rawdata[:256*3]
    # Compressed stream of pixels.
    rawpixels = rawdata

    palette = default_palette

    # Our framebuffer where the compressed image will be uncompresed.
    pixels = bytearray(320*200)
    pointer = 0

    from jetpyck.gfxdat import BitGenerator
    stream = BitGenerator(rawpixels)
    while pointer < len(pixels):
        is_repeating = stream.get_bit()
        qty = 1
        if is_repeating:
            qty = 2 + stream.get_bits(5)
        colorindex = stream.get_byte()
        for i in range(qty):
            pixels[pointer] = colorindex
            pointer += 1
            if pointer >= len(pixels):
                break

    if (remainder := stream.remaining_bits()) >= 8:
        print("Warning: {} trailing bytes ({} bits) are being ignored after the pixel data.".format(remainder // 8, remainder))

    img = Image.new(mode="P", size=(320, 200))
    img.putpalette(palette)
    img.putdata(pixels)

    return img

In [ ]:
default_palette = gfxdat_parser((BASEDIR / '_JETP_A0.DAT').read_bytes()).getpalette()

In [ ]:
for f in BASEDIR.glob("_jetp_?0.dat", case_sensitive=False):
    display(gfxdat_parser(f.read_bytes()))

In [ ]:
for filename in sorted(glob.iglob("/home/denilson/Games/*/JETPACK?.DAT")):
    print(filename)
    with open(filename, "rb") as f:
        if "xjetpack" in filename:
            img = xmas_gfxdat_parser(f.read())
        else:
            img = gfxdat_parser(f.read())
        display(img)